# Hello Agent - Microsoft Agent Framework Demo

Welcome to the Hello Agent notebook! This simple demo showcases the Microsoft Agent Framework by creating an agent with basic tools.

## What you'll learn:
- Set up Microsoft Agent Framework with Azure OpenAI
- Create simple tools (calculator and summarizer)  
- Test the agent with different prompts

**Prerequisites**: 
- Azure OpenAI resource with deployed chat model
- Environment variables configured (see next cell)

In [1]:
# 1. Setup and Imports
import os
from typing import Annotated
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from pydantic import Field
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("Required environment variables:")
print(f"AZURE_OPENAI_ENDPOINT: {'OK' if os.getenv('AZURE_OPENAI_ENDPOINT') else 'MISSING'}")
print(f"AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: {'OK' if os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME') else 'MISSING'}")
print(f"AZURE_OPENAI_API_VERSION: {'OK' if os.getenv('AZURE_OPENAI_API_VERSION') else 'MISSING'}")

# Create Azure OpenAI client
try:
    client = AzureOpenAIChatClient(
        endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
        deployment_name=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'),
        api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-02-01'),
        credential=DefaultAzureCredential()
    )
    print("\nAzure OpenAI client created successfully!")
except Exception as e:
    print(f"\nError: {e}")
    print("Make sure to run 'az login' and check your environment variables.")
    client = None

Required environment variables:
AZURE_OPENAI_ENDPOINT: OK
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: OK
AZURE_OPENAI_API_VERSION: OK

Azure OpenAI client created successfully!

Azure OpenAI client created successfully!


### Environment Setup (.env file)
If you see missing variables above, create a `.env` file with:
```
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=your-chat-deployment-name
AZURE_OPENAI_API_VERSION=2024-02-01
```
Then authenticate with: `az login`

In [2]:
# 2. Create Agent Tools
def calculator(
    expression: Annotated[str, Field(description="Math expression like '2+2' or '10*5'")]
) -> str:
    """Calculate mathematical expressions."""
    try:
        # Safety check - only allow basic math
        allowed = set('0123456789+-*/.() ')
        if not all(c in allowed for c in expression):
            return "Error: Only basic math operations allowed"
        
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {str(e)}"

def summarizer(
    text: Annotated[str, Field(description="Text to summarize")]
) -> str:
    """Summarize text by taking key sentences."""
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    if len(sentences) <= 2:
        return text
    return f"{sentences[0]}. {sentences[-1]}."

# Test the tools
print("Testing tools:")
print("Calculator:", calculator("2 + 3 * 4"))
print("Summarizer:", summarizer("This is a test. It has multiple sentences. This is the end."))

Testing tools:
Calculator: 2 + 3 * 4 = 14
Summarizer: This is a test. This is the end.


In [3]:
# 3. Create the Hello Agent
agent = ChatAgent(
    chat_client=client,
    name="HelloAgent",
    instructions="""You are HelloAgent, a friendly AI assistant. 

You have two tools:
- Calculator: for math expressions
- Summarizer: for summarizing text

Be helpful and use tools when appropriate.""",
    tools=[calculator, summarizer]
)

print("HelloAgent created with 2 tools!")
print(f"Agent name: {agent.name}")
print("Ready to chat!")

HelloAgent created with 2 tools!
Agent name: HelloAgent
Ready to chat!


In [4]:
# 4. Test Basic Conversation
print("=== Basic Chat Test ===")

# Check if agent exists and is properly configured
if 'agent' not in globals():
    print("Error: Agent not found. Please run cell 5 (Create the Hello Agent) first!")
elif client is None:
    print("Error: Azure OpenAI client is None. Please check your environment setup in cell 2!")
else:
    # Test 1: Simple greeting
    print("\nUser: Hello! What's your name?")
    try:
        response = await agent.run("Hello! What's your name?")
        print(f"HelloAgent: {response}")
    except Exception as e:
        print(f"Error: {e}")

    # Test 2: Ask about tools
    print("\nUser: What tools do you have available?")
    try:
        response = await agent.run("What tools do you have available?")
        print(f"HelloAgent: {response}")
    except Exception as e:
        print(f"Error: {e}")

    print("\nBasic chat test completed!")

=== Basic Chat Test ===

User: Hello! What's your name?
HelloAgent: Hi — I'm HelloAgent, your friendly AI assistant. How can I help you today?

User: What tools do you have available?
HelloAgent: Hi — I'm HelloAgent, your friendly AI assistant. How can I help you today?

User: What tools do you have available?
HelloAgent: I have the following tools available:

- Calculator: evaluate math expressions (e.g., "2+2", "10*5", "sqrt(2)" where supported).
- Summarizer: create concise summaries by extracting key sentences from provided text.
- Multi-tool parallel runner: can run the above tools in parallel when appropriate.

Tell me what you want to do (a calculation, a text to summarize, or both), and I’ll use the appropriate tool.

Basic chat test completed!
HelloAgent: I have the following tools available:

- Calculator: evaluate math expressions (e.g., "2+2", "10*5", "sqrt(2)" where supported).
- Summarizer: create concise summaries by extracting key sentences from provided text.
- Multi-t

In [5]:
# 5. Test Tool Usage
async def test_tools():
    print("=== Tool Usage Test ===")
    
    test_cases = [
        "Calculate 25 + 17",
        "What's 144 divided by 12?",
        "Please summarize: The Microsoft Agent Framework is a powerful SDK for building AI agents. It provides simple APIs for creating conversational AI applications. The framework integrates with Azure OpenAI and supports function calling for tool usage."
    ]
    
    for i, prompt in enumerate(test_cases, 1):
        print(f"\n--- Test {i} ---")
        print(f"User: {prompt}")
        try:
            response = await agent.run(prompt)
            print(f"HelloAgent: {response}")
        except Exception as e:
            print(f"Error: {e}")

await test_tools()

=== Tool Usage Test ===

--- Test 1 ---
User: Calculate 25 + 17
HelloAgent: 42

--- Test 2 ---
User: What's 144 divided by 12?
HelloAgent: 42

--- Test 2 ---
User: What's 144 divided by 12?
HelloAgent: 144 divided by 12 is 12.

--- Test 3 ---
User: Please summarize: The Microsoft Agent Framework is a powerful SDK for building AI agents. It provides simple APIs for creating conversational AI applications. The framework integrates with Azure OpenAI and supports function calling for tool usage.
HelloAgent: 144 divided by 12 is 12.

--- Test 3 ---
User: Please summarize: The Microsoft Agent Framework is a powerful SDK for building AI agents. It provides simple APIs for creating conversational AI applications. The framework integrates with Azure OpenAI and supports function calling for tool usage.
HelloAgent: The Microsoft Agent Framework is an SDK for building AI agents. It offers simple APIs for conversational applications, integrates with Azure OpenAI, and supports function-calling for t

## Try Your Own Prompts
Use the cell below to chat with HelloAgent directly. Try asking for calculations or text summaries!

In [6]:
# 6. Interactive Chat - Easy to Modify
# Simply change the message below and run the cell to test different prompts:

message = "Calculate (10 + 5) * 3"  # Change this message to test different prompts

print(f"You: {message}")
try:
    response = await agent.run(message)
    print(f"HelloAgent: {response}")
except Exception as e:
    print(f"Error: {e}")

print("\n" + "="*50)
print("TIP: To test other prompts, change the 'message' variable above and re-run this cell")
print("Try these examples:")
print('message = "What tools do you have?"')
print('message = "What\'s 100 divided by 4?"') 
print('message = "Summarize: AI is transforming industries. ML enables data analysis. AI will continue evolving."')
print('message = "Hello, how are you today?"')

You: Calculate (10 + 5) * 3
HelloAgent: 45

TIP: To test other prompts, change the 'message' variable above and re-run this cell
Try these examples:
message = "What tools do you have?"
message = "What's 100 divided by 4?"
message = "Summarize: AI is transforming industries. ML enables data analysis. AI will continue evolving."
message = "Hello, how are you today?"
HelloAgent: 45

TIP: To test other prompts, change the 'message' variable above and re-run this cell
Try these examples:
message = "What tools do you have?"
message = "What's 100 divided by 4?"
message = "Summarize: AI is transforming industries. ML enables data analysis. AI will continue evolving."
message = "Hello, how are you today?"


## Summary

Congratulations! You've successfully created HelloAgent with the Microsoft Agent Framework!

### What you built:
- Azure OpenAI integration
- Calculator tool for math operations  
- Summarizer tool for text processing
- Simple conversational agent

### Key concepts learned:
- Setting up Agent Framework with Azure OpenAI
- Creating function tools with type annotations
- Tool registration and automatic function calling
- Basic agent testing and interaction

### Next steps:
- Add more sophisticated tools (weather, web search, etc.)
- Implement conversation memory
- Deploy to production with Azure services
- Explore multi-agent scenarios

**Resources:**
- [Microsoft Agent Framework Docs](https://learn.microsoft.com/en-us/agent-framework/)
- [Azure OpenAI Service](https://learn.microsoft.com/en-us/azure/ai-services/openai/)